# Experiment 2: rectangular EVA02-B/14 with attribute queries

Run all cells in Kaggle with **GPU and Internet enabled**. This is the next experiment after the 224-square attribute-query notebook. It changes input geometry to **280 high x 140 wide** while retaining the same pretrained backbone, head, loss, augmentation operations, effective batch, optimizer, schedule, and seed. This trains a fresh model from generic pretrained weights; it does not resume a square model.

**Correction to the experiment plan:** with 14 px patches, 280 x 140 gives **20 x 10 = 200 patch tokens**, compared with 256 for 224 x 224. This run changes shape and token count together. For a closer compute comparison, also run **308 x 168** (22 x 12 = 264 tokens). Exact compute equality is not claimed. Dimensions throughout are **height, width**. Direct resizing makes the crop taller; it does not preserve every source crop's exact aspect ratio.

Default: 40 learned queries; decoder width 256, eight heads, one layer; equally averaged global/query logits; eight epochs; microbatch 32 with two accumulation steps (effective batch 64). Validation and submission both use horizontal-flip logit averaging. Set `HEAD_MODE` to match the square run being compared. No result or score improvement has been measured.

Both dimensions must be divisible by 14. timm loads/interpolates pretrained absolute position embeddings for the rectangular grid. The offline implementation saves the explicit height/width and rebuilds axis-specific rotary embeddings; it never infers a square grid from token count. Grid-equivalence checks run before training, followed by full offline/timm equivalence with nonzero classifiers. Packaging rechecks trained predictions and attribute order.

Outputs include the best checkpoint, offline submission ZIP, per-domain epoch history, validation logits/labels, and configuration with annotation hashes and organizer revision. Names contain geometry and head mode, so these outputs do not overwrite the square experiment. All PAR fitting uses the organizer's training split; validation is held out and hidden test images are never accessed.

References: [Query2Label](https://github.com/SlongLiu/query2labels), [timm v1.0.22 EVA](https://github.com/huggingface/pytorch-image-models/blob/v1.0.22/timm/models/eva.py), [timm positional embeddings](https://github.com/huggingface/pytorch-image-models/blob/v1.0.22/timm/layers/pos_embed_sincos.py).


## 1. Download the provided development data

These cells adapt the baseline's Market1501, PA100K, and PETA download workflow. Existing complete downloads are reused. Annotation discovery supports both the historical flat layout and the organizer's task-specific layout. An existing local clone is reused without silently updating it; its commit is recorded with the run.


In [ ]:
%pip -q install gdown tqdm requests timm==1.0.22


In [ ]:
import csv
import shutil
import subprocess
import zipfile
from pathlib import Path

import gdown
import requests
from tqdm.auto import tqdm

WORK = Path('/kaggle/working')
REPO = WORK / 'UPAR-Challenge-2027'
DATA = WORK / 'upar_2027' / 'data'
DATA.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/speckean/UPAR-Challenge-2027.git', str(REPO)
    ], check=True)
if not (REPO / 'peta_file_mapping.txt').is_file():
    raise FileNotFoundError(f'Missing PETA mapping in {REPO}; attach the organizer starter kit')
for split in ('train', 'val'):
    candidates = [
        REPO / 'data' / 'annotations' / 'task1' / split / 'gt.csv',
        REPO / 'data' / 'annotations' / f'{split}.csv',
    ]
    original = next((p for p in candidates if p.is_file()), None)
    if original is None:
        raise FileNotFoundError(f'Cannot locate {split} annotations: {candidates}')
    destination = DATA / 'annotations' / f'{split}.csv'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(original, destination)
    print(f'{split}: {original}')
print('Data directory:', DATA)


In [ ]:
# The annotation paths determine exactly which images must be available.
needed = {'Market1501': set(), 'PA100k': set(), 'PETA': set()}
for split in ('train', 'val'):
    with (DATA / 'annotations' / f'{split}.csv').open(newline='') as handle:
        for row in list(csv.reader(handle))[1:]:
            relative = Path(row[0])
            needed[relative.parts[0]].add(relative)

def missing(source):
    return [path for path in needed[source] if not (DATA / path).is_file()]

def check_zip(path):
    if path.exists() and not zipfile.is_zipfile(path):
        path.unlink()  # Remove an interrupted download before retrying.
    return path.exists()

def require_complete(source):
    absent = missing(source)
    if absent:
        raise RuntimeError(f'{source}: {len(absent)} images missing; examples: {absent[:5]}')
    print(f'{source}: {len(needed[source]):,} annotated images available')

print({source: len(paths) for source, paths in needed.items()})

## Market1501

In [ ]:
market_archive = DATA / 'market_1501.zip'
market_dir = DATA / 'Market1501'
if missing('Market1501'):
    if not check_zip(market_archive):
        result = gdown.download(
            'https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ',
            output=str(market_archive), quiet=False, use_cookies=False, fuzzy=True
        )
        if not result or not zipfile.is_zipfile(market_archive):
            raise RuntimeError('Market1501 download failed or is not a ZIP file')
    with zipfile.ZipFile(market_archive) as archive:
        archive.extractall(DATA)
    extracted = DATA / 'Market-1501-v15.09.15'
    if extracted.exists():
        shutil.copytree(extracted, market_dir, dirs_exist_ok=True)
        shutil.rmtree(extracted)
require_complete('Market1501')
market_archive.unlink(missing_ok=True)

## PA-100K

In [ ]:
pa_dir = DATA / 'PA100k'
pa_dir.mkdir(exist_ok=True)
pa_archive = pa_dir / 'data.zip'
if missing('PA100k'):
    if not check_zip(pa_archive):
        gdown.download_folder(
            'https://drive.google.com/drive/folders/1d_D0Yh7C262gr0ef9EqkvG_M3fqgAWa2?usp=sharing',
            output=str(pa_dir), quiet=False, use_cookies=False
        )
        if not zipfile.is_zipfile(pa_archive):
            raise RuntimeError('PA-100K data.zip was not downloaded')
    with zipfile.ZipFile(pa_archive) as archive:
        archive.extractall(pa_dir)
require_complete('PA100k')
pa_archive.unlink(missing_ok=True)

## PETA

Only the mapped images are extracted from the archive, directly to their challenge filenames.

In [ ]:
peta_dir = DATA / 'PETA'
peta_dir.mkdir(exist_ok=True)
peta_archive = peta_dir / 'peta.zip'
if missing('PETA'):
    if not check_zip(peta_archive):
        url = 'https://www.dropbox.com/s/52ylx522hwbdxz6/PETA.zip?dl=1'
        temporary = peta_archive.with_suffix('.zip.part')
        with requests.get(url, stream=True, timeout=(30, 120)) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            with temporary.open('wb') as output, tqdm(total=total or None, unit='B', unit_scale=True, desc='PETA') as bar:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        output.write(chunk)
                        bar.update(len(chunk))
        temporary.replace(peta_archive)
        if not zipfile.is_zipfile(peta_archive):
            raise RuntimeError('PETA download is not a ZIP file')
    with (REPO / 'peta_file_mapping.txt').open(newline='') as handle:
        mapping = list(csv.reader(handle))
    with zipfile.ZipFile(peta_archive) as archive:
        members = set(archive.namelist())
        for source, destination in tqdm(mapping, desc='Extracting PETA images'):
            target = DATA / destination
            if target.is_file():
                continue
            member = source.removeprefix('PETA/')
            if member not in members:
                raise FileNotFoundError(f'PETA archive is missing {member}')
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as original, target.open('wb') as output:
                shutil.copyfileobj(original, output)
require_complete('PETA')
peta_archive.unlink(missing_ok=True)

In [ ]:
for source in needed:
    require_complete(source)
print('Ready:', DATA)
print('Train annotations:', DATA / 'annotations' / 'train.csv')
print('Validation annotations:', DATA / 'annotations' / 'val.csv')

## 2. Configuration: height first, width second

The default is `IMAGE_HEIGHT = 280`, `IMAGE_WIDTH = 140` (200 patch tokens). Use 308 and 168 for the near-token-count control (264 tokens), or 224 and 224 for a square control with this exact code path. Each geometry/head combination gets a separate output prefix. Preserve outputs when rerunning the same combination with other settings.

Keep `HEAD_MODE`, loss, effective batch 64, eight epochs, learning rate, layer decay, and seed matched to the square run. If memory is tight, use microbatch 16 with four accumulation steps. Higher resolution and aspect-preserving letterboxing are separate experiments. This notebook uses the baseline's direct-resize, flip, padding/crop, and erasing operations; random crop now targets the configured rectangular dimensions.


In [ ]:
"""Kaggle training script for UPAR 2027 Track 1 with EVA02-B/14 and an attribute-query decoder.

Reads challenge data from /kaggle/working/upar_2027/data by default.
This file is also the source for the training cells in the accompanying notebook.
"""

import csv
import importlib.util
import json
import hashlib
import platform
import math
import os
import random
import shutil
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


DATA_ROOT = Path(os.environ.get("UPAR_DATA_ROOT", "/kaggle/working/upar_2027/data"))
OUTPUT_ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
HEAD_MODE = "hybrid"  # "global", "query", or "hybrid"; use separate training runs.
IMAGE_HEIGHT = 280
IMAGE_WIDTH = 140
IMAGE_SIZE = (IMAGE_HEIGHT, IMAGE_WIDTH)  # Height, width; not PIL's width, height.
PATCH_SIZE = 14
if any(size <= 0 or size % PATCH_SIZE for size in IMAGE_SIZE):
    raise ValueError("Both input dimensions must be positive multiples of 14")
PATCH_GRID = tuple(size // PATCH_SIZE for size in IMAGE_SIZE)
MODEL_TAG = f"upar_track1_eva02_b14_attribute_query_{IMAGE_HEIGHT}x{IMAGE_WIDTH}_{HEAD_MODE}"
print(f"Input HxW={IMAGE_SIZE}; patch grid={PATCH_GRID}; patch tokens={math.prod(PATCH_GRID)}")
DECODER_DIM = 256
DECODER_HEADS = 8
DECODER_LAYERS = 1
DECODER_DROPOUT = 0.1
GLOBAL_LOGIT_WEIGHT = 0.5  # Hybrid only; fixed before evaluating.
ACCUMULATION_STEPS = 2
INFERENCE_BATCH_SIZE = 32
ARCHITECTURE = "eva02_base_patch14_448"  # timm EVA02
PRETRAINED_TAG = "mim_in22k_ft_in22k_in1k"  # ImageNet-22k MIM pretraining, ImageNet-22k then ImageNet-1k fine-tuning.
EPOCHS = 8
WARMUP_EPOCHS = 1
BATCH_SIZE = 32  # Effective batch 64 with two accumulation steps.
NUM_WORKERS = 4
LEARNING_RATE = 1e-4  # Peak learning rate of the classifier head.
LAYER_DECAY = 0.75  # Each earlier transformer block gets this factor less learning rate.
WEIGHT_DECAY = 0.05
DROP_PATH = 0.1  # Stochastic depth rate.
USE_IMAGENET_WEIGHTS = True  # Needs Internet once (Hugging Face Hub), or a cached timm checkpoint.
# IMAGE_SIZE is the (height, width) tuple defined above.
MEAN = (0.48145466, 0.4578275, 0.40821073)  # CLIP normalization used by EVA02.
STD = (0.26862954, 0.26130258, 0.27577711)
SEED = 42


## 3. Read annotations and images

The released training CSV header omits the image-column label; this parser handles both train and validation formats.

In [ ]:
def read_annotations(path):
    with path.open(newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        # The released train.csv omits the image-column name in its header.
        names = header if len(header) == 40 else header[1:]
        names = [name.removeprefix("# ").strip() for name in names]
        if len(names) != 40:
            raise ValueError(f"Expected 40 attributes in {path}, got {len(names)}")
        paths, labels = [], []
        for row_number, row in enumerate(reader, 2):
            if len(row) != 41:
                raise ValueError(f"{path}:{row_number}: expected image path and 40 labels")
            paths.append(row[0])
            labels.append([int(value) for value in row[1:]])
    labels = np.asarray(labels, dtype=np.float32)
    if not np.isin(labels, [0, 1]).all():
        raise ValueError(f"{path} contains labels other than 0 and 1")
    return names, paths, labels


class AttributeDataset(Dataset):
    def __init__(self, root, paths, labels, transform):
        self.root, self.paths, self.labels, self.transform = root, paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.root / self.paths[index]) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, torch.from_numpy(self.labels[index])




## 4. Shared query head, training model, and metrics

The decoder receives patch tokens (the class token is excluded), normalizes/projects them to width 256, and lets 40 learned queries attend to them. Each attribute has its own readout. The original global classifier is retained. Global/query logits are equally averaged in hybrid mode. Both readouts start at zero, as in the original global classifier.

The exact query-head/wrapper source below is also embedded in `run.py`, avoiding separate training and inference implementations. Layer-wise learning-rate decay applies to the encoder; the decoder and global classifier get the head learning rate. No weight decay is applied to attribute queries, biases, or normalization vectors.

The compact self-check exercises decoder learning and layer-group coverage before downloading pretrained weights. The full packaged EVA02/query model is checked against timm after training, before a ZIP is accepted.


In [ ]:
ATTRIBUTE_HEAD_PY = '''
class AttributeQueryHead(nn.Module):
    def __init__(self, input_dim, num_attributes=40, decoder_dim=256,
                 decoder_heads=8, decoder_layers=1, decoder_dropout=0.1):
        super().__init__()
        if decoder_dim % decoder_heads or decoder_layers < 1:
            raise ValueError("Decoder width must divide into heads; use at least one layer")
        self.input_norm = nn.LayerNorm(input_dim, eps=1e-6)
        self.projection = nn.Linear(input_dim, decoder_dim)
        self.queries = nn.Parameter(torch.empty(num_attributes, decoder_dim))
        nn.init.normal_(self.queries, std=0.02)
        self.layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=decoder_dim, nhead=decoder_heads,
                dim_feedforward=decoder_dim * 4, dropout=decoder_dropout,
                activation="gelu", batch_first=True, norm_first=True,
            ) for _ in range(decoder_layers)
        ])
        self.output_norm = nn.LayerNorm(decoder_dim, eps=1e-6)
        self.readout_weight = nn.Parameter(torch.zeros(num_attributes, decoder_dim))
        self.readout_bias = nn.Parameter(torch.zeros(num_attributes))

    def forward(self, patches):
        memory = self.projection(self.input_norm(patches))
        queries = self.queries.unsqueeze(0).expand(patches.shape[0], -1, -1)
        for layer in self.layers:
            queries = layer(queries, memory)
        features = self.output_norm(queries)
        return (features * self.readout_weight.unsqueeze(0)).sum(-1) + self.readout_bias


class EVA02AttributeQuery(nn.Module):
    def __init__(self, backbone, num_attributes=40, decoder_dim=256,
                 decoder_heads=8, decoder_layers=1, decoder_dropout=0.1,
                 head_mode="hybrid", global_logit_weight=0.5):
        super().__init__()
        if head_mode not in ("global", "query", "hybrid"):
            raise ValueError(f"Unknown head mode: {head_mode}")
        if not 0.0 <= global_logit_weight <= 1.0:
            raise ValueError("Global logit weight must be in [0, 1]")
        self.backbone = backbone
        self.head_mode = head_mode
        self.global_logit_weight = global_logit_weight
        self.attribute_head = AttributeQueryHead(
            backbone.num_features, num_attributes, decoder_dim, decoder_heads,
            decoder_layers, decoder_dropout,
        )

    def forward(self, images, return_branches=False):
        tokens = self.backbone.forward_features(images)
        global_logits = self.backbone.forward_head(tokens)
        if self.head_mode == "global":
            combined, query_logits = global_logits, None
        else:
            patches = tokens[:, self.backbone.num_prefix_tokens:]
            query_logits = self.attribute_head(patches)
            if self.head_mode == "query":
                combined = query_logits
            else:
                weight = self.global_logit_weight
                combined = weight * global_logits + (1.0 - weight) * query_logits
        if return_branches:
            return {"combined": combined, "global": global_logits, "query": query_logits}
        return combined
'''
exec(ATTRIBUTE_HEAD_PY)

def model_config():
    return dict(num_attributes=40, decoder_dim=DECODER_DIM,
                decoder_heads=DECODER_HEADS, decoder_layers=DECODER_LAYERS,
                decoder_dropout=DECODER_DROPOUT, head_mode=HEAD_MODE,
                global_logit_weight=GLOBAL_LOGIT_WEIGHT)


def build_model(pretrained):
    backbone = timm.create_model(
        f"{ARCHITECTURE}.{PRETRAINED_TAG}", pretrained=pretrained, num_classes=40,
        img_size=IMAGE_SIZE, drop_path_rate=DROP_PATH,
    )
    nn.init.zeros_(backbone.head.weight)
    nn.init.zeros_(backbone.head.bias)
    return EVA02AttributeQuery(backbone, **model_config())


def parameter_groups(model):
    depth = len(model.backbone.blocks)
    groups = {}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        local = name.removeprefix("backbone.")
        lid = depth + 1
        if name.startswith("backbone."):
            if local.startswith(("cls_token", "pos_embed", "patch_embed")):
                lid = 0
            elif local.startswith("blocks."):
                lid = int(local.split(".")[1]) + 1
        no_decay = parameter.ndim == 1 or name in (
            "backbone.cls_token", "backbone.pos_embed", "attribute_head.queries",
        )
        key = (lid, no_decay)
        if key not in groups:
            groups[key] = {"params": [], "weight_decay": 0.0 if no_decay else WEIGHT_DECAY,
                           "lr_scale": LAYER_DECAY ** (depth + 1 - lid)}
        groups[key]["params"].append(parameter)
    return list(groups.values())


def check_query_head():
    # Tiny CPU check: both readout and query/memory paths must learn.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(SEED)
        head = AttributeQueryHead(32, decoder_dim=32, decoder_heads=4, decoder_dropout=0.0)
        patches = torch.randn(2, 9, 32, requires_grad=True)
        labels = torch.randint(0, 2, (2, 40)).float()
        optimizer = torch.optim.SGD(head.parameters(), lr=0.1)
        for _ in range(2):
            optimizer.zero_grad(set_to_none=True)
            patches.grad = None
            logits = head(patches)
            assert logits.shape == (2, 40) and torch.isfinite(logits).all()
            nn.functional.binary_cross_entropy_with_logits(logits, labels).backward()
            optimizer.step()
        assert head.queries.grad is not None and head.queries.grad.abs().sum() > 0
        assert patches.grad is not None and patches.grad.abs().sum() > 0
        assert all(p.grad is None or torch.isfinite(p.grad).all() for p in head.parameters())
    print("Query-head shape, gradient, and learning checks passed")

def challenge_metrics(probabilities, labels, sources):
    """mA, instance F1 and their harmonic mean, averaged over source datasets."""
    eps = 1e-20
    results = []
    for source in sorted(set(sources)):
        rows = np.asarray([s == source for s in sources])
        pred = probabilities[rows] > 0.5
        gt = labels[rows] > 0.5
        tp = (pred & gt).sum(0)
        tn = (~pred & ~gt).sum(0)
        positives, negatives = gt.sum(0), (~gt).sum(0)
        label_ma = (tp / (positives + eps) + tn / (negatives + eps)) / 2
        intersection = (pred & gt).sum(1)
        precision = (intersection / (pred.sum(1) + eps)).mean()
        recall = (intersection / (gt.sum(1) + eps)).mean()
        # As in the original evaluation script: F1 of the image-averaged precision and recall.
        ma, f1 = label_ma.mean(), 2 * precision * recall / (precision + recall + eps)
        results.append((ma, f1, 2 * ma * f1 / (ma + f1 + eps)))
    ma, f1, avg = np.mean(results, axis=0)
    return {"mA": ma, "Instance_F1": f1, "Challenge_Avg": avg}

check_query_head()


In [ ]:
def evaluate(model, loader, device, criterion):
    model.eval()
    losses, count = 0.0, 0
    collected = {"combined": [], "global": [], "query": []}
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                original = model(images, return_branches=True)
                flipped = model(torch.flip(images, dims=[3]), return_branches=True)
            for key in collected:
                if original[key] is not None:
                    logits = (original[key].float() + flipped[key].float()) / 2
                    collected[key].append(logits.cpu().numpy())
                    if key == "combined":
                        losses += criterion(logits, labels).item() * len(images)
            count += len(images)
    logits = {key: np.concatenate(rows) for key, rows in collected.items() if rows}
    if count == 0:
        raise ValueError("Validation loader is empty")
    return losses / count, logits


def scores_from_logits(logits, labels, sources):
    probabilities = torch.from_numpy(logits).sigmoid().numpy()
    overall = {key: float(value) for key, value in
               challenge_metrics(probabilities, labels, sources).items()}
    per_source = {}
    sources_array = np.asarray(sources)
    for source in sorted(set(sources)):
        rows = sources_array == source
        per_source[source] = {key: float(value) for key, value in
                              challenge_metrics(probabilities[rows], labels[rows], sources_array[rows]).items()}
    return overall, per_source


def main():
    if not 0 <= WARMUP_EPOCHS < EPOCHS:
        raise ValueError("Use fewer warm-up epochs than total epochs")
    if BATCH_SIZE < 1 or ACCUMULATION_STEPS < 1:
        raise ValueError("Batch size and accumulation steps must be positive")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.backends.cudnn.benchmark = True
    names, train_paths, train_labels = read_annotations(DATA_ROOT / "annotations/train.csv")
    val_names, val_paths, val_labels = read_annotations(DATA_ROOT / "annotations/val.csv")
    if names != val_names or len(set(names)) != 40:
        raise ValueError("Attribute names must be unique and match across splits")
    if set(train_paths) & set(val_paths):
        raise ValueError("Training and validation image paths overlap")
    val_sources = [Path(path).parts[0] for path in val_paths]
    print(f"Loaded {len(train_paths):,} train and {len(val_paths):,} validation images")
    normalize = transforms.Normalize(MEAN, STD)
    train_transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE), transforms.RandomHorizontalFlip(),
        transforms.Pad(10, padding_mode="edge"), transforms.RandomCrop(IMAGE_SIZE),
        transforms.ToTensor(), normalize, transforms.RandomErasing(p=0.25, value="random"),
    ])
    val_transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE), transforms.ToTensor(), normalize,
    ])
    train_loader = DataLoader(
        AttributeDataset(DATA_ROOT, train_paths, train_labels, train_transform),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(), drop_last=True,
    )
    val_loader = DataLoader(
        AttributeDataset(DATA_ROOT, val_paths, val_labels, val_transform),
        batch_size=INFERENCE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    if not len(train_loader):
        raise ValueError("Training split is smaller than one microbatch")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(USE_IMAGENET_WEIGHTS).to(device)
    check_runtime_equivalence(model, device)
    print(f"{MODEL_TAG}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
    print(f"Microbatch={BATCH_SIZE}; effective batch={BATCH_SIZE * ACCUMULATION_STEPS}")
    groups = parameter_groups(model)
    grouped = [id(p) for group in groups for p in group["params"]]
    assert len(grouped) == len(set(grouped)) == len(list(model.parameters()))
    positives = train_labels.sum(axis=0)
    ratio = (len(train_labels) - positives) / np.maximum(positives, 1)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(
        np.clip(ratio, 1, 20), dtype=torch.float32, device=device,
    ))
    optimizer = torch.optim.AdamW(groups, lr=LEARNING_RATE)
    steps_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
    total_steps, warmup_steps = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch

    def learning_rate(step):
        if step < warmup_steps:
            return LEARNING_RATE * (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return LEARNING_RATE * 0.5 * (1 + math.cos(math.pi * progress))

    def sha256(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()

    revision = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                              capture_output=True, text=True).stdout.strip()
    run_config = dict(
        model_tag=MODEL_TAG, architecture=ARCHITECTURE, pretrained_tag=PRETRAINED_TAG,
        model_config=model_config(), epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS,
        batch_size=BATCH_SIZE, accumulation_steps=ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE, layer_decay=LAYER_DECAY, weight_decay=WEIGHT_DECAY,
        drop_path=DROP_PATH, image_size=IMAGE_SIZE, patch_grid=PATCH_GRID, patch_tokens=math.prod(PATCH_GRID), seed=SEED, mean=MEAN, std=STD,
        validation_tta="horizontal_flip_logit_average", data_revision=revision,
        train_sha256=sha256(DATA_ROOT / "annotations/train.csv"),
        val_sha256=sha256(DATA_ROOT / "annotations/val.csv"),
        torch_version=str(torch.__version__), timm_version=timm.__version__,
        python_version=platform.python_version(), cuda_version=torch.version.cuda,
    )
    (OUTPUT_ROOT / f"{MODEL_TAG}_config.json").write_text(json.dumps(run_config, indent=2))
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    checkpoint_path = OUTPUT_ROOT / f"{MODEL_TAG}_best.pth"
    best_score, step, history = -1.0, 0, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_total = 0.0
        for batch_index, (images, labels) in enumerate(train_loader):
            for group in optimizer.param_groups:
                group["lr"] = learning_rate(step) * group["lr_scale"]
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            group_start = (batch_index // ACCUMULATION_STEPS) * ACCUMULATION_STEPS
            group_size = min(ACCUMULATION_STEPS, len(train_loader) - group_start)
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                loss = criterion(model(images), labels)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite training loss at epoch {epoch}, batch {batch_index}")
            scaler.scale(loss / group_size).backward()
            train_total += loss.item() * len(images)
            if (batch_index + 1) % ACCUMULATION_STEPS == 0 or batch_index + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1
        val_loss, branch_logits = evaluate(model, val_loader, device, criterion)
        metrics, per_source = scores_from_logits(branch_logits["combined"], val_labels, val_sources)
        record = dict(epoch=epoch, train_bce=train_total / (len(train_loader) * BATCH_SIZE),
                      val_bce=val_loss, metrics=metrics, per_source=per_source,
                      branches={key: scores_from_logits(value, val_labels, val_sources)[0]
                                for key, value in branch_logits.items()})
        history.append(record)
        (OUTPUT_ROOT / f"{MODEL_TAG}_history.json").write_text(json.dumps(history, indent=2))
        print(f"Epoch {epoch}/{EPOCHS}: train BCE={record['train_bce']:.4f}, val BCE={val_loss:.4f}, "
              f"mA={metrics['mA']:.4f}, Inst_F1={metrics['Instance_F1']:.4f}, "
              f"Challenge_Avg={metrics['Challenge_Avg']:.4f}")
        print("Per-source:", per_source)
        print("Branch diagnostics (jointly trained):", record["branches"])
        if metrics["Challenge_Avg"] > best_score:
            best_score = metrics["Challenge_Avg"]
            torch.save({
                "state_dict": {key: (value.detach().half() if value.is_floating_point()
                                     else value.detach()).cpu()
                               for key, value in model.state_dict().items()},
                "architecture": ARCHITECTURE, "model_config": model_config(),
                "num_heads": model.backbone.blocks[0].attn.num_heads,
                "attribute_names": names, "image_size": IMAGE_SIZE, "patch_grid": PATCH_GRID,
                "epoch": epoch, "val_loss": val_loss, "val_metrics": metrics,
                "run_config": run_config,
            }, checkpoint_path)
            np.savez_compressed(
                OUTPUT_ROOT / f"{MODEL_TAG}_best_val_logits.npz", **branch_logits,
                labels=val_labels, image_paths=np.asarray(val_paths),
                sources=np.asarray(val_sources), attribute_names=np.asarray(names),
            )
            print("Saved", checkpoint_path)
    # Free optimizer/encoder GPU allocations before loading the packaged model and parity reference.
    del model, optimizer, scaler, groups, group, loss, images, labels
    if device.type == "cuda":
        torch.cuda.empty_cache()
    package_submission(checkpoint_path, [DATA_ROOT / path for path in val_paths[:4]], names)


## 5. Rectangular offline inference and equivalence checks

`run.py` reads the explicit image dimensions from the checkpoint. Absolute position embeddings are loaded strictly from the fine-tuned state; rotary positions are regenerated independently for the height and width axes against EVA02's reference 16 x 16 grid. The packaged implementation accepts the configured shape only, catching reversed dimensions early.

A CPU check compares rectangular rotary embeddings with pinned timm on several grids, including the square control. The full model is compared before training with nonzero classifiers. After training, packaging checks real-image predictions, single/batch consistency, and attribute reordering. No timm installation is needed in the evaluation container.


In [ ]:
RUN_PY = '''"""UPAR 2027 Track 1 inference entry point (EVA02-B/14 attribute queries).

timm is not installed in the evaluation image, so EVA02 is defined here in plain PyTorch.
Parameter names match timm's Eva, so the fine-tuned state dict loads strictly.
"""
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torchvision import transforms

HERE = Path(__file__).resolve().parent
CHECKPOINT = HERE / "assets/__MODEL_TAG___best.pth"
MEAN = (0.48145466, 0.4578275, 0.40821073)  # CLIP normalization used by EVA02.
STD = (0.26862954, 0.26130258, 0.27577711)
BATCH_SIZE = 32
MODEL = None
ATTRIBUTE_NAMES = None
TRANSFORM = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class PatchEmbed(nn.Module):
    def __init__(self, patch_size, dim):
        super().__init__()
        self.proj = nn.Conv2d(3, dim, patch_size, patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


def rotary_embedding(grid_size, head_dim, ref_grid_size=16, temperature=10000.0):
    """timm RotaryEmbeddingCat (in_pixels=False): sin and cos of 2D patch positions, concatenated."""
    num_bands = head_dim // 4
    bands = 1.0 / temperature ** (torch.arange(num_bands, dtype=torch.float32) / num_bands)
    # Scale each axis independently to EVA02's reference 16 x 16 grid.
    height, width = (int(value) for value in grid_size)
    if min(height, width) <= 0:
        raise ValueError("Rotary grid dimensions must be positive")
    ys = torch.arange(height, dtype=torch.float32) / height * ref_grid_size
    xs = torch.arange(width, dtype=torch.float32) / width * ref_grid_size
    grid = torch.stack(torch.meshgrid(ys, xs, indexing="ij"), dim=-1).unsqueeze(-1)
    positions = (grid * bands).reshape(height * width, -1)
    return torch.cat([positions.sin().repeat_interleave(2, -1), positions.cos().repeat_interleave(2, -1)], -1)


def apply_rotary(x, embedding):
    sin, cos = embedding.chunk(2, -1)
    rotated = torch.stack([-x[..., 1::2], x[..., ::2]], -1).reshape(x.shape)  # (x0, x1) -> (-x1, x0)
    return x * cos + rotated * sin


class Attention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, rope):
        b, n, c = x.shape
        q, k, v = (
            projection(x).reshape(b, n, self.num_heads, -1).transpose(1, 2)
            for projection in (self.q_proj, self.k_proj, self.v_proj)
        )
        # Rotary position embeddings rotate the patch tokens only, not the class token.
        q = torch.cat([q[:, :, :1], apply_rotary(q[:, :, 1:], rope)], dim=2).type_as(v)
        k = torch.cat([k[:, :, :1], apply_rotary(k[:, :, 1:], rope)], dim=2).type_as(v)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(b, n, c))


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1_g = nn.Linear(dim, hidden_dim)
        self.fc1_x = nn.Linear(dim, hidden_dim)
        self.act = nn.SiLU()
        self.norm = nn.LayerNorm(hidden_dim, eps=1e-6)
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.norm(self.act(self.fc1_g(x)) * self.fc1_x(x)))


class Block(nn.Module):
    def __init__(self, dim, num_heads, hidden_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = SwiGLU(dim, hidden_dim)

    def forward(self, x, rope):
        x = x + self.attn(self.norm1(x), rope)
        return x + self.mlp(self.norm2(x))


class EVA02(nn.Module):
    """timm Eva for EVA02: RoPE, SwiGLU with sub-LayerNorm, average pooling of patch tokens."""

    def __init__(self, patch_size, grid_size, dim, depth, num_heads, hidden_dim, num_classes):
        super().__init__()
        self.grid_size = tuple(int(value) for value in grid_size)
        num_patches = math.prod(self.grid_size)
        self.image_size = tuple(value * patch_size for value in self.grid_size)
        self.num_features = dim
        self.num_prefix_tokens = 1
        self.patch_embed = PatchEmbed(patch_size, dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, dim))
        self.blocks = nn.ModuleList([Block(dim, num_heads, hidden_dim) for _ in range(depth)])
        self.fc_norm = nn.LayerNorm(dim, eps=1e-6)
        self.head = nn.Linear(dim, num_classes)
        self.register_buffer("rope", rotary_embedding(self.grid_size, dim // num_heads), persistent=False)

    def forward_features(self, x):
        if tuple(x.shape[-2:]) != self.image_size:
            raise ValueError(f"Expected HxW={self.image_size}, got {tuple(x.shape[-2:])}")
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(len(x), -1, -1), x], dim=1) + self.pos_embed
        for block in self.blocks:
            x = block(x, self.rope)
        return x

    def forward_head(self, tokens):
        return self.head(self.fc_norm(tokens[:, 1:].mean(dim=1)))

    def forward(self, x):
        return self.forward_head(self.forward_features(x))


__ATTRIBUTE_HEAD_SOURCE__

def load_model():
    global MODEL, ATTRIBUTE_NAMES, TRANSFORM
    checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    ATTRIBUTE_NAMES = checkpoint["attribute_names"]
    size = tuple(int(value) for value in checkpoint["image_size"])
    if len(size) != 2 or any(value <= 0 for value in size):
        raise ValueError("Checkpoint image_size must contain positive height and width")
    TRANSFORM = transforms.Compose([
        transforms.Resize(size), transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    ])
    state = {key.removeprefix("backbone."): value for key, value in checkpoint["state_dict"].items()
             if key.startswith("backbone.")}
    patch_size = state["patch_embed.proj.weight"].shape[-1]
    if any(value % patch_size for value in size):
        raise ValueError("Checkpoint image size must divide into full patches")
    grid_size = tuple(value // patch_size for value in size)
    if math.prod(grid_size) != state["pos_embed"].shape[1] - 1:
        raise ValueError("Checkpoint positional tokens disagree with its HxW grid")
    if grid_size != tuple(checkpoint["patch_grid"]):
        raise ValueError("Checkpoint patch grid disagrees with image_size")
    backbone = EVA02(
        patch_size=patch_size,
        grid_size=grid_size,
        dim=state["pos_embed"].shape[2],
        depth=len({key.split(".")[1] for key in state if key.startswith("blocks.")}),
        num_heads=int(checkpoint["num_heads"]),
        hidden_dim=state["blocks.0.mlp.fc1_g.weight"].shape[0],
        num_classes=state["head.weight"].shape[0],
    )
    model = EVA02AttributeQuery(backbone, **checkpoint["model_config"])
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    MODEL = model.float().to(DEVICE).eval()

def predict_batch(samples):
    if MODEL is None:
        load_model()
    images = []
    for sample in samples:
        with Image.open(sample["image_path"]) as image:
            images.append(TRANSFORM(image.convert("RGB")))
    batch = torch.stack(images).to(DEVICE)
    with torch.inference_mode(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        logits = MODEL(batch).float() + MODEL(torch.flip(batch, dims=[3])).float()
    probabilities = (logits / 2).sigmoid().clamp(0.0, 1.0).cpu().numpy()
    probabilities = np.nan_to_num(probabilities, nan=0.0)
    rows = []
    for sample, row in zip(samples, probabilities):
        order = [ATTRIBUTE_NAMES.index(name) for name in sample["attribute_names"]]
        rows.append(row[order].astype(float).tolist())
    return rows


def predict_image(sample):
    return predict_batch([sample])[0]
'''

RUN_PY = RUN_PY.replace('__ATTRIBUTE_HEAD_SOURCE__', ATTRIBUTE_HEAD_PY).replace('__MODEL_TAG__', MODEL_TAG)

METADATA_YAML = '''# Code submissions are executed by the challenges ingestion program, which
# imports run.py and calls predict_image() / predict_batch(). Nothing in this
# file is used for inference, it exists only so the platform never treats the
# archive as a result submission. Leave it as-is.
command: python3 -c "print('Submission package ready. The ingestion program imports run.py')"
'''


def package_submission(checkpoint_path, sample_images, names):
    folder = OUTPUT_ROOT / f"{MODEL_TAG}_submission"
    assets = folder / "assets"
    assets.mkdir(parents=True, exist_ok=True)
    (folder / "run.py").write_text(RUN_PY)
    (folder / "metadata.yaml").write_text(METADATA_YAML)
    shutil.copy2(checkpoint_path, assets / checkpoint_path.name)
    archive_path = OUTPUT_ROOT / f"{MODEL_TAG}_submission.zip"
    expected = {"run.py", "metadata.yaml", f"assets/{checkpoint_path.name}"}
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in sorted(expected):
            archive.write(folder / name, name)
    with zipfile.ZipFile(archive_path) as archive:
        print("ZIP contents:", archive.namelist())
        assert set(archive.namelist()) == expected
        with tempfile.TemporaryDirectory() as temp_dir:
            archive.extractall(temp_dir)
            spec = importlib.util.spec_from_file_location("upar_submission_check", Path(temp_dir) / "run.py")
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            module.load_model()
            samples = [
                {"index": i, "image": str(path), "image_path": str(path), "attribute_names": names}
                for i, path in enumerate(sample_images)
            ]
            batch = np.asarray(module.predict_batch(samples))
            single = np.asarray(module.predict_image(samples[0]))
            assert batch.shape == (len(samples), 40) and single.shape == (40,)
            assert np.isfinite(batch).all() and ((batch >= 0) & (batch <= 1)).all()
            print("Packaged inference smoke test passed")
            shuffled = list(reversed(names))
            reordered = np.asarray(module.predict_image({**samples[0], "attribute_names": shuffled}))
            assert np.allclose(single, batch[0], atol=1e-3, rtol=1e-3)
            assert np.allclose(reordered, single[::-1], atol=1e-3, rtol=1e-3)
            # Full encoder, decoder, branch fusion, and real-image logits must match timm.
            reference = build_model(False).to(module.DEVICE).eval()
            reference.load_state_dict(module.MODEL.state_dict())
            inputs = torch.stack([module.TRANSFORM(Image.open(path).convert("RGB"))
                                  for path in sample_images[:2]]).to(module.DEVICE)
            with torch.inference_mode():
                difference = (reference(inputs) - module.MODEL(inputs)).abs().max().item()
            assert difference < 2e-3, f"run.py and timm outputs differ by {difference}"
            print(f"run.py matches timm (max logit difference {difference:.1e})")
            assert "__ATTRIBUTE_HEAD_SOURCE__" not in (Path(temp_dir) / "run.py").read_text()
    print("Checkpoint:", checkpoint_path)
    print(f"ZIP size: {archive_path.stat().st_size / 1e6:.0f} MB")
    print("Upload this ZIP to Codabench:", archive_path)

def check_runtime_equivalence(reference, device):
    """Fail before training if timm and the offline implementation disagree."""
    was_training = reference.training
    reference.eval()
    # Nonzero readouts ensure the check actually exercises features and attention.
    readouts = [reference.backbone.head.weight, reference.backbone.head.bias,
                reference.attribute_head.readout_weight, reference.attribute_head.readout_bias]
    saved = [parameter.detach().clone() for parameter in readouts]
    try:
        with torch.random.fork_rng(devices=[device.index or 0] if device.type == "cuda" else []):
            torch.manual_seed(SEED)
            with torch.no_grad():
                for parameter in readouts:
                    parameter.normal_(std=0.02)
            with tempfile.TemporaryDirectory() as directory:
                path = Path(directory) / "run.py"
                path.write_text(RUN_PY)
                spec = importlib.util.spec_from_file_location("attribute_query_preflight", path)
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                backbone = reference.backbone
                offline_backbone = module.EVA02(
                    patch_size=backbone.patch_embed.proj.weight.shape[-1],
                    grid_size=tuple(backbone.patch_embed.grid_size),
                    dim=backbone.num_features, depth=len(backbone.blocks),
                    num_heads=backbone.blocks[0].attn.num_heads,
                    hidden_dim=backbone.blocks[0].mlp.fc1_g.weight.shape[0], num_classes=40,
                )
                offline = module.EVA02AttributeQuery(offline_backbone, **model_config()).to(device).eval()
                offline.load_state_dict(reference.state_dict(), strict=True)
                inputs = torch.randn(2, 3, *IMAGE_SIZE, device=device)
                with torch.inference_mode():
                    # Compare all branches even when the selected ablation skips one.
                    expected_tokens = reference.backbone.forward_features(inputs)
                    actual_tokens = offline.backbone.forward_features(inputs)
                    assert torch.allclose(expected_tokens, actual_tokens, atol=2e-3, rtol=1e-3)
                    expected_global = reference.backbone.forward_head(expected_tokens)
                    actual_global = offline.backbone.forward_head(actual_tokens)
                    expected_query = reference.attribute_head(expected_tokens[:, 1:])
                    actual_query = offline.attribute_head(actual_tokens[:, 1:])
                    assert torch.allclose(expected_global, actual_global, atol=2e-3, rtol=1e-3)
                    assert torch.allclose(expected_query, actual_query, atol=2e-3, rtol=1e-3)
                    difference = (reference(inputs) - offline(inputs)).abs().max().item()
                    assert difference < 2e-3, f"Offline implementation mismatch: {difference}"
                del offline, offline_backbone
        print(f"Pre-training offline/timm equivalence passed: {difference:.1e}")
    finally:
        with torch.no_grad():
            for parameter, value in zip(readouts, saved):
                parameter.copy_(value)
        reference.train(was_training)


def check_rectangular_positions():
    """Check axes/order against pinned timm without downloading model weights."""
    from timm.layers.pos_embed_sincos import build_rotary_pos_embed
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "run.py"
        path.write_text(RUN_PY)
        spec = importlib.util.spec_from_file_location("rectangular_grid_check", path)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        for grid in ((16, 16), (20, 10), (22, 12), (24, 16), PATCH_GRID):
            sin, cos = build_rotary_pos_embed(
                list(grid), dim=64, in_pixels=False, ref_feat_shape=(16, 16),
            )
            expected = torch.cat([sin, cos], dim=-1)
            actual = module.rotary_embedding(grid, 64)
            assert actual.shape == (math.prod(grid), 128)
            assert torch.allclose(actual, expected, atol=1e-6, rtol=1e-6), grid
    print("Square and rectangular rotary-grid equivalence checks passed")


check_rectangular_positions()


## 6. Train and compare with the square experiment

Run the final cell. The default output prefix is `upar_track1_eva02_b14_attribute_query_280x140_hybrid`. Compare its per-domain Challenge_Avg against the square experiment with the same head mode and training settings. The hybrid run's global branch is jointly trained and is not an independent global-only control.

Recommended sequence: square 224 x 224; tall 280 x 140; near-compute control 308 x 168. The first tall run has fewer patch tokens, so its result cannot isolate shape from compute. If both tall runs improve, test 336 x 224 for additional detail. Set the dimensions in section 2 and rerun all definition/training cells or restart the kernel; model/output configuration must stay consistent.

The notebook saves per-domain history and best-validation logits. These are development scores, not evidence of hidden-domain gains; source-held-out assessment remains useful. No measured improvement is claimed. Preserve every run's ZIP/configuration. The leaderboard counts the latest valid submission.


In [ ]:
main()